In [0]:
# Silver layer - clean and enrich the bronze data

import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, TimestampType
from pyspark.sql.window import Window

bronze_df = spark.table("finops_catalog.focus_billing_schema.bronze_focus_billing")
print(f"Bronze records: {bronze_df.count():,}")

# Clean up 'NULL' string literals

def cleanse_null_strings(df, columns):
    """Replace 'NULL' string literals with actual NULL values"""
    for col in columns:
        df = df.withColumn(col, 
            F.when(F.col(col) == "NULL", None)
             .when(F.col(col) == "null", None)
             .when(F.trim(F.col(col)) == "", None)
             .otherwise(F.col(col)))
    return df

string_columns = [field.name for field in bronze_df.schema.fields 
                  if str(field.dataType) == "StringType"]
silver_df = cleanse_null_strings(bronze_df, string_columns)

print("Cleaned 'NULL' string literals")

# Cast cost columns to numeric

cost_columns = [col for col in bronze_df.columns if "cost" in col.lower()]
for cost_col in cost_columns:
    silver_df = silver_df.withColumn(
        cost_col, 
        F.expr(f"try_cast({cost_col} as double)"))

print(f"Cast {len(cost_columns)} cost columns to double")

# Cast quantity columns to numeric

quantity_columns = ["ConsumedQuantity", "PricingQuantity"]
for qty_col in quantity_columns:
    silver_df = silver_df.withColumn(
        qty_col,
        F.expr(f"try_cast({qty_col} as double)"))

print(f"Cast quantity columns to double")

# Add derived metrics for analysis

silver_df = silver_df.withColumn(
    "effective_unit_price",
    F.when(F.col("ConsumedQuantity") > 0, 
           F.col("EffectiveCost") / F.col("ConsumedQuantity"))
     .otherwise(None)
).withColumn(
    "discount_amount",
    F.col("ListCost") - F.col("EffectiveCost")
).withColumn(
    "discount_percentage",
    F.when(F.col("ListCost") > 0,
           ((F.col("ListCost") - F.col("EffectiveCost")) / F.col("ListCost")) * 100)
     .otherwise(0)
).withColumn(
    "charge_duration_hours",
    (F.unix_timestamp("ChargePeriodEnd") - 
     F.unix_timestamp("ChargePeriodStart")) / 3600
).withColumn(
    "is_commitment_discount_applied",
    F.when(F.col("CommitmentDiscountId").isNotNull(), True)
     .otherwise(False)
).withColumn(
    "cost_category",
    F.when(F.col("BilledCost") == 0, "Free/No Charge")
     .when(F.col("BilledCost") < 0.01, "Micro (<$0.01)")
     .when(F.col("BilledCost") < 1, "Small (<$1)")
     .when(F.col("BilledCost") < 100, "Medium (<$100)")
     .when(F.col("BilledCost") < 1000, "Large (<$1K)")
     .otherwise("Very Large (>=$1K)")
)

print("Added derived metrics")

# Add time dimensions

silver_df = silver_df.withColumn(
    "charge_date",
    F.to_date("ChargePeriodStart")
).withColumn(
    "charge_year",
    F.year("ChargePeriodStart")
).withColumn(
    "charge_month",
    F.month("ChargePeriodStart")
).withColumn(
    "charge_day",
    F.dayofmonth("ChargePeriodStart")
).withColumn(
    "charge_hour",
    F.hour("ChargePeriodStart")
).withColumn(
    "charge_day_of_week",
    F.dayofweek("ChargePeriodStart")
).withColumn(
    "charge_week_of_year",
    F.weekofyear("ChargePeriodStart")
).withColumn(
    "charge_quarter",
    F.quarter("ChargePeriodStart")
).withColumn(
    "charge_year_month",
    F.date_format("ChargePeriodStart", "yyyy-MM")
)

print("Added time dimensions")

# Add data quality flags

silver_df = silver_df.withColumn(
    "has_resource_id",
    F.col("ResourceId").isNotNull()
).withColumn(
    "has_complete_cost_data",
    (F.col("BilledCost").isNotNull()) & 
    (F.col("EffectiveCost").isNotNull())
).withColumn(
    "is_valid_time_range",
    F.col("ChargePeriodEnd") >= F.col("ChargePeriodStart")
).withColumn(
    "data_quality_score",
    (
        F.when(F.col("Id").isNotNull(), 20).otherwise(0) +
        F.when(F.col("ServiceName").isNotNull(), 20).otherwise(0) +
        F.when(F.col("has_resource_id"), 20).otherwise(0) +
        F.when(F.col("has_complete_cost_data"), 20).otherwise(0) +
        F.when(F.col("is_valid_time_range"), 20).otherwise(0)
    )
)

print("Added data quality flags")

# Classify resources into service families

silver_df = silver_df.withColumn(
    "service_family",
    F.when(F.col("ServiceName").contains("Compute"), "Compute")
     .when(F.col("ServiceName").contains("Storage") | 
           F.col("ServiceName").contains("S3"), "Storage")
     .when(F.col("ServiceName").contains("Database") | 
           F.col("ServiceName").contains("RDS"), "Database")
     .when(F.col("ServiceName").contains("Network") | 
           F.col("ServiceName").contains("VPC") |
           F.col("ServiceName").contains("CloudFront"), "Networking")
     .when(F.col("ServiceName").contains("Lambda"), "Serverless")
     .otherwise("Other")
).withColumn(
    "is_on_demand",
    ~F.col("is_commitment_discount_applied")
)

print("Added resource classification")

# Remove duplicates
initial_count = silver_df.count()
silver_df = silver_df.dropDuplicates(["Id"])
final_count = silver_df.count()
duplicates_removed = initial_count - final_count
print(f"Removed {duplicates_removed} duplicates")

# Add audit metadata

silver_df = silver_df.withColumn(
    "silver_processed_at",
    F.current_timestamp()
).withColumn(
    "silver_processing_date",
    F.current_date()
).withColumn(
    "data_source",
    F.lit("bronze_focus_billing")
).withColumn(
    "processing_version",
    F.lit("v1.0")
)

print("Added audit metadata")

# Get final statistics
final_stats = silver_df.agg(
    F.count("*").alias("total_records"),
    F.sum("BilledCost").alias("total_billed_cost"),
    F.sum("EffectiveCost").alias("total_effective_cost"),
    F.sum("discount_amount").alias("total_savings"),
    F.avg("data_quality_score").alias("avg_quality_score"),
    F.countDistinct("ServiceName").alias("unique_services"),
    F.countDistinct("ResourceId").alias("unique_resources")
).collect()[0]

print("\nSilver transformation summary:")
print(f"  Records: {final_stats['total_records']:,}")
print(f"  Total Billed: ${final_stats['total_billed_cost']:,.2f}")
print(f"  Total Effective: ${final_stats['total_effective_cost']:,.2f}")
print(f"  Total Savings: ${final_stats['total_savings']:,.2f}")
print(f"  Avg Quality Score: {final_stats['avg_quality_score']:.1f}/100")
print(f"  Unique Services: {final_stats['unique_services']}")
print(f"  Unique Resources: {final_stats['unique_resources']}")
print(f"  Total Columns: {len(silver_df.columns)}")

# ============================================================================
# 11. WRITE TO SILVER TABLE
# ============================================================================

silver_table = "finops_catalog.focus_billing_schema.silver_focus_billing"
silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(silver_table)

print(f"\n✅ Silver table written: {silver_table}")
print("\n🎯 Transformation complete! Data ready for gold layer and analytics.")